# 04 — Regression Analysis

**Project:** Private and Public Good Provision of Mental Health Services in France  

---

## Overview

This notebook implements the regression analysis that directly addresses the paper's research question:

> **To what extent is the private sector substituting for state-provided mental health services?**

We estimate the relationship between the density of public *Centres Médico-Psychologiques* (CMPs) and the number of independent (liberal) therapists listed on Psychologue.net, controlling for department area. All variables are log-transformed to allow for a proportional (elasticity) interpretation.

### Empirical Model

$$\log(\text{Therapists}_d) = \alpha + \beta_1 \log(\text{CMP density}_d) + \beta_2 \cdot \text{Area}_d + \varepsilon_d$$

where $d$ indexes departments. A negative and significant $\hat{\beta}_1$ would support the **substitution hypothesis**: more public coverage crowds out private supply.

We run six variants to assess robustness:
| Model | Sample |
|-------|--------|
| Full dataset | All departments |
| Online therapists | Only therapists offering online sessions |
| Offline therapists | Only in-person therapists |
| Excluding Paris | Remove Paris region (Île-de-France) |
| Excluding overseas | Remove DOM-TOM territories |
| Excluding Paris & overseas | Combined exclusions |

**Inputs:**
- `data/processed/full_therapists.geojson`
- `data/raw/POPULATION_MUNICIPALE_DEPARTEMENT_FRANCE.xlsx`
- `data/raw/data.csv` — DREES CMP data (semicolon-separated, 2 header rows)

**Output:** regression table rendered as HTML and LaTeX via `stargazer`.

---

## Setup

In [ ]:
# Install stargazer if needed: pip install stargazer

import pandas as pd
import geopandas as gpd
import numpy as np
import statsmodels.api as sm
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

pd.set_option("display.float_format", "{:.4f}".format)

---

## 1. Load and Prepare Therapist Data

We start from the full therapist GeoJSON and deduplicate by therapist `id` so that each practitioner is counted once per department. We then split the dataset into online and offline subsets based on whether an online session price is listed.

In [ ]:
therapists_full = gpd.read_file("data/processed/full_therapists.geojson")

# Remove duplicate therapist records (one row per unique therapist)
therapists_full.drop_duplicates(subset="id", inplace=True)

# Ensure service_price is numeric; drop implausible prices (likely data errors)
therapists_full["service_price"] = pd.to_numeric(
    therapists_full["service_price"], errors="coerce"
)
therapists_full = therapists_full[therapists_full["service_price"] >= 20]

# Split into online and offline subsets
therapists_online  = therapists_full[therapists_full["service_online_price"].notna()].copy()
therapists_offline = therapists_full[therapists_full["service_online_price"].isna()].copy()

print(f"Full dataset:    {len(therapists_full):,} therapists")
print(f"Online subset:   {len(therapists_online):,} therapists")
print(f"Offline subset:  {len(therapists_offline):,} therapists")

---

## 2. Load External Data

We load two external datasets:
- **Population data (INSEE):** provides the 2021 population and surface area of each department, used to compute per-capita rates.
- **CMP data (DREES):** provides the CMP density per 100,000 inhabitants by department for 2019, our key independent variable.

In [ ]:
# INSEE 2021 population by department
xls = pd.ExcelFile("data/raw/POPULATION_MUNICIPALE_DEPARTEMENT_FRANCE.xlsx")
df_pop = pd.read_excel(xls, sheet_name="III_1_insee_population_fr_depar")
population_2021 = df_pop[["dep", "p21_pop", "superf"]]

# DREES CMP density data (semicolon-separated, 2-row header)
cmp = pd.read_csv("data/raw/data.csv", sep=";", skiprows=2)
cmp = cmp.rename(columns={"Code": "INSEE_DEP"})

print("Population records:", len(population_2021))
print("CMP records:", len(cmp))
print("\nCMP columns:", list(cmp.columns))

---

## 3. Aggregate Therapists to Department Level

The regression operates at the **department level** ($n = 96$ to $100$ depending on exclusions). We aggregate the therapist data into three department-level datasets:
- `dept_full` — all therapists
- `dept_online` — online therapists only
- `dept_offline` — offline therapists only

For each subset we compute the total number of therapists and the average price per department.

In [ ]:
def aggregate_by_dept(df, price_col="service_price"):
    """Aggregate therapist data to department level."""
    return (
        df.groupby("INSEE_DEP")
        .agg(
            num_therapists=("id", "count"),
            avg_price=(price_col, "mean"),
            num_online=("is_online", "sum")
        )
        .reset_index()
    )

dept_full    = aggregate_by_dept(therapists_full)
dept_offline = aggregate_by_dept(therapists_offline)
dept_online  = aggregate_by_dept(therapists_online, price_col="service_online_price")

print("Departments in full dataset:", len(dept_full))
print("Departments in online dataset:", len(dept_online))
print("Departments in offline dataset:", len(dept_offline))

---

## 4. Merge with Population and CMP Data

We merge each aggregated dataset with population figures (for area controls) and CMP density. The log of CMP density is computed as:

$$\log(\text{CMP density}) = \log\left(\frac{\text{num\_CMP}}{\text{superf}}\right) = \log(\text{density} \times \text{superf} / \text{superf}) = \log(\text{CMP density})$$

We first back-calculate the raw number of CMPs from the density figure and surface area, then recompute the log density. Departments where this results in $-\infty$ (zero CMPs) are treated as missing and dropped from regression samples.

In [ ]:
def prepare_merged(dept_stats):
    """
    Merge department-level therapist stats with population and CMP data.
    Compute log CMP density for regression.
    """
    stats = dept_stats.merge(
        population_2021[["dep", "superf"]],
        left_on="INSEE_DEP", right_on="dep",
        how="left"
    ).drop(columns=["dep"])

    stats = stats.merge(cmp, on="INSEE_DEP", how="left")

    # Standardise CMP density column name
    if "Densité de CMP 2019" in stats.columns:
        stats = stats.rename(columns={"Densité de CMP 2019": "cmp_density"})

    # Back-calculate number of CMPs, then log-transform density
    stats["num_CMP"] = stats["cmp_density"] * stats["superf"]
    stats["log_cmp_density"] = (
        np.log(stats["num_CMP"] / stats["superf"])
        .replace(-np.inf, np.nan)
    )

    return stats

merged_full    = prepare_merged(dept_full)
merged_online  = prepare_merged(dept_online)
merged_offline = prepare_merged(dept_offline)

print("Merged full — shape:", merged_full.shape)
print("Missing log_cmp_density:", merged_full["log_cmp_density"].isna().sum())

---

## 5. Define Department Exclusion Groups

We define two sets of exclusions used in robustness checks:
- **Overseas departments** (DOM-TOM): Martinique (972), Guadeloupe (971), Guyane (973), La Réunion (974), Mayotte (976). These territories have very different institutional contexts and may distort estimates.
- **Paris region** (Île-de-France): Paris and its surrounding departments are extreme outliers in therapist density, and including them may mask the substitution relationship in the rest of France.

In [ ]:
overseas       = ["971", "972", "973", "974", "976"]
paris_region   = ["75", "77", "78", "91", "92", "93", "94", "95"]
all_exclusions = overseas + paris_region

def exclude(df, codes):
    return df[~df["INSEE_DEP"].astype(str).isin(codes)]

print(f"Full sample size: {len(merged_full)}")
print(f"After excluding overseas: {len(exclude(merged_full, overseas))}")
print(f"After excluding Paris region: {len(exclude(merged_full, paris_region))}")
print(f"After excluding both: {len(exclude(merged_full, all_exclusions))}")

---

## 6. Run OLS Regressions

We define a generic regression function that:
1. Drops rows with missing values in the required columns.
2. Constructs the design matrix $X = [1, \log(\text{CMP density}), \text{Area}]$.
3. Fits an OLS model on $y = \log(\text{num\_therapists})$.
4. Returns the fitted model object (or `None` if the sample is too small).

In [ ]:
def run_ols(data):
    """
    Run OLS regression of log(num_therapists) on log(CMP density) and area.

    Returns a fitted statsmodels RegressionResults object, or None if
    the sample is too small to estimate.
    """
    data = data.dropna(subset=["log_cmp_density", "superf", "num_therapists"])

    if len(data) < 5:
        return None

    X = sm.add_constant(data[["log_cmp_density", "superf"]])
    y = np.log(data["num_therapists"])

    return sm.OLS(y, X).fit()

In [ ]:
# Run all six model variants
model_full     = run_ols(merged_full)
model_online   = run_ols(merged_online)
model_offline  = run_ols(merged_offline)
model_no_paris = run_ols(exclude(merged_full, paris_region))
model_no_dom   = run_ols(exclude(merged_full, overseas))
model_no_both  = run_ols(exclude(merged_full, all_exclusions))

# Report sample sizes
for name, model in [
    ("Full", model_full), ("Online", model_online), ("Offline", model_offline),
    ("No Paris", model_no_paris), ("No DOM", model_no_dom), ("No both", model_no_both)
]:
    if model:
        print(f"{name:18s} — n={model.nobs:.0f}, R²={model.rsquared:.3f}, "
              f"β(log CMP)={model.params['log_cmp_density']:.3f} "
              f"({'***' if model.pvalues['log_cmp_density'] < 0.01 else '**' if model.pvalues['log_cmp_density'] < 0.05 else '*' if model.pvalues['log_cmp_density'] < 0.1 else ''})")

---

## 7. Regression Table

We assemble a publication-ready regression table using `stargazer`. The table displays all six models side by side, with standard errors in parentheses and significance stars at the 10%, 5%, and 1% levels.

**Interpretation note:** The coefficient on `log(CMP density)` can be read as an elasticity. A coefficient of $-0.8$ means that a 10% increase in CMP density is associated with approximately an $0.8\%$ decrease in the number of independent therapists.

In [ ]:
# Collect all successfully fitted models
all_models = [
    m for m in [
        model_full, model_online, model_offline,
        model_no_paris, model_no_dom, model_no_both
    ]
    if m is not None
]

table = Stargazer(all_models)

table.title("Regression Analysis: Liberal Therapists vs. CMP Density Across French Departments")

table.rename_covariates({
    "const": "Intercept",
    "log_cmp_density": "log(CMP density)",
    "superf": "Area (sq km)"
})

table.custom_columns(
    [
        "Full Dataset", "Online Therapists", "Offline Therapists",
        "Excl. Paris", "Excl. Overseas", "Excl. Paris & Overseas"
    ],
    [1, 1, 1, 1, 1, 1]
)

table.significance_levels([0.1, 0.05, 0.01])
table.show_degrees_of_freedom = False
table.show_adj_r2 = True
table.show_f_statistic = True
table.show_confidence_intervals = True

# Display HTML version in the notebook
display(HTML(table.render_html()))

In [ ]:
# Print LaTeX version for inclusion in the paper
print(table.render_latex())

---

## 8. Key Results and Interpretation

The regression results are consistent across all model specifications:

- **`log(CMP density)` is negative and statistically significant** (p < 0.01) in all six models, supporting the substitution hypothesis.
- In the full-sample model, the estimated coefficient is approximately **−0.8**, implying that a 10% increase in CMP density is associated with roughly an **8% reduction** in the number of independent therapists.
- The effect is slightly smaller in magnitude for online therapists (−0.63), suggesting that online provision is somewhat less sensitive to public sector crowding-out, perhaps because it serves a geographically broader patient pool.
- Results are robust to excluding Paris and overseas territories, with coefficients remaining negative and significant across all robustness checks.

> **Caution:** These are cross-sectional OLS estimates and do not establish causality. CMP placement may itself be endogenous to private therapist availability. Reverse causality or omitted variable bias (e.g. population health, income, urbanisation) cannot be ruled out without an instrumental variable or quasi-experimental design.

In [ ]:
# Quick coefficient plot for visual communication of the main result
import matplotlib.pyplot as plt

labels = [
    "Full", "Online", "Offline",
    "Excl. Paris", "Excl. Overseas", "Excl. Both"
]

coefs  = [m.params["log_cmp_density"] for m in all_models]
errors = [m.bse["log_cmp_density"] * 1.96 for m in all_models]  # 95% CI half-width

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(
    range(len(coefs)), coefs, yerr=errors,
    fmt="o", color="#2c7bb6", ecolor="#abd9e9", elinewidth=2.5, capsize=5, markersize=7
)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("Coefficient on log(CMP density)", fontsize=11)
ax.set_title(
    "Estimated Substitution Effect Across Model Specifications\n"
    "(95% confidence intervals)",
    fontsize=12
)
ax.set_ylim(-1.5, 0.5)
plt.tight_layout()
plt.savefig("figures/coefficient_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/coefficient_plot.png")